<a href="https://colab.research.google.com/github/stefisha/genai24/blob/main/grounding_dino_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Intitial setup

In [ ]:
import os
HOME = os.getcwd()
print(HOME)
%cd {HOME}
!git clone https://github.com/IDEA-Research/GroundingDINO.git
%cd {HOME}/GroundingDINO
!pip install datumaro
!pip install -r requirements.txt
!pip install -q -e .

Verify that configuration is correct

In [ ]:
import os

CONFIG_PATH = os.path.join(HOME, "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py")
print(CONFIG_PATH, "; exist:", os.path.isfile(CONFIG_PATH))

## Download Grounding DINO weights

In [ ]:
%cd {HOME}
!mkdir {HOME}/weights
%cd {HOME}/weights

!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth

In [ ]:
import os

WEIGHTS_NAME = "groundingdino_swint_ogc.pth"
WEIGHTS_PATH = os.path.join(HOME, "weights", WEIGHTS_NAME)
print(WEIGHTS_PATH, "; exist:", os.path.isfile(WEIGHTS_PATH))

##Download image to be annotated

In [ ]:
%cd {HOME}
!mkdir {HOME}/data
%cd {HOME}/data

#!wget -q https://media.istockphoto.com/id/1319636830/vector/set-of-wooden-chairs-for-outdoor-and-indoor-usage-retro-chair-without-cushion-flat-vector.jpg?s=612x612&w=0&k=20&c=mQeYCC0ryra8IdCVmYHnxSiZUpP8NdQF5OKandWtrI4=

##Load Grounding DINO model

In [ ]:
%cd {HOME}/GroundingDINO

from groundingdino.util.inference import load_model, load_image, predict, annotate

model = load_model(CONFIG_PATH, WEIGHTS_PATH)

##Using the model

We will first perform inference on the downloaded images, then save the annotations to be exported in COCO format. Tweak the text prompts to get to your desired output.

In [ ]:
import os
import supervision as sv
import torch
from datumaro import Polygon, Bbox, RleMask, Dataset, DatasetItem, Image


images = ["cessna-185-engine-parts.jpg"]
results = []
for image_name in images:
  image_path = os.path.join(HOME, "data", image_name)

  TEXT_PROMPT = "red pipe"
  BOX_TRESHOLD = 0.35
  TEXT_TRESHOLD = 0.25

  image_source, image = load_image(image_path)
  # Generate predictions
  boxes, logits, phrases = predict(
      model=model,
      image=image,
      caption=TEXT_PROMPT,
      box_threshold=BOX_TRESHOLD,
      text_threshold=TEXT_TRESHOLD
  )

  annotated_frame = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)

  %matplotlib inline
  sv.plot_image(annotated_frame, (16, 16))


  h, w, _ = image_source.shape
  # Assemble annotations to be used by datumaro
  annotations = []
  for box in boxes:
    result = box * torch.Tensor([w, h, w, h])
    annotations.append(Bbox(result[0], result[1], result[2], result[3], label=1))

  results.append(
      DatasetItem(
        id=image_name.split(".")[0],
        media=Image(path=image_path),
        annotations=annotations
        )
      )
dataset = Dataset.from_iterable(results,categories = [TEXT_PROMPT])


Export from datumaro

In [ ]:
dataset.export("./coco_export.json",format="coco")

You may now download the exported COCO file for training, or run a validation workflow to refine the accuracy of the annotations.

In [ ]:
while True:
    pass